In [3]:
import pandas as pd
from easynmt import EasyNMT
from tqdm import tqdm; tqdm.pandas()
from ast import literal_eval
import numpy as np
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch.nn.functional as F
from transformers import pipeline
import torch

In [4]:
df = pd.read_csv("df2.csv")

## Vertaling

In [7]:
# Deze code is bedoeld om zinnen van de ene taal naar de andere te vertalen.  
# Het vertaalmodel vertaalt zinnen van het Nederlands naar het Engels.  
# De zinnen worden eerst in een lijst gezet zodat ze makkelijk verwerkt kunnen worden.  
# Elke zin in de lijst wordt vertaald.  
# De vertalingen van alle zinnen worden verzameld en teruggegeven.  


model = EasyNMT("opus-mt", use_cuda=True)
model.device = 'cuda'

def get_translation(sentences, source_lang="nl", target_lang="en"):
    sentences = literal_eval(sentences)
    translations = []
    for s in sentences:
        translation = model.translate(s, source_lang=source_lang, target_lang=target_lang)
        translations.append(translation)
    return translations

In [8]:
# Deze code vertaalt de zinnen die het woord "POLICE" bevatten naar het Engels.  
 
df["en_police_sentences"] = df.police_sentences.progress_apply(lambda x: get_translation(x))

  0%|          | 0/5412 [00:00<?, ?it/s]/home/rdubel/.local/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
100%|██████████| 5412/5412 [58:17<00:00,  1.55it/s]  


## Sentiment berekenen

In [15]:
# Deze code haalt een model op dat sentimentanalyse kan uitvoeren, specifiek gericht op aspecten in teksten.  

absa_tokenizer = AutoTokenizer.from_pretrained("yangheng/deberta-v3-base-absa-v1.1")
absa_model = AutoModelForSequenceClassification.from_pretrained("yangheng/deberta-v3-base-absa-v1.1").to("cuda")

/home/rdubel/.local/lib/python3.8/site-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(
/home/rdubel/.local/lib/python3.8/site-packages/transformers/convert_slow_tokenizer.py:551: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


In [16]:
# Het model bepaalt de waarschijnlijkheden voor drie mogelijke gevoelens: negatief, neutraal en positief.  

def get_aspect_based_sentiment(sentence, aspect="POLICE"):
    inputs = absa_tokenizer(f"[CLS] {sentence} [SEP] {aspect} [SEP]", return_tensors="pt").to("cuda")
    outputs = absa_model(**inputs)
    probs = F.softmax(outputs.logits, dim=1)
    probs = probs.detach().cpu().numpy()[0]
    result = {"negative": probs[0], "neutral": probs[1], "positive": probs[2]}
    return result

In [17]:
# Deze code analyseert het sentiment voor een lijst van zinnen, specifiek gericht op het aspect "POLICE".  

def get_sentence_sentiment(sentences):
    results = []
    sentences = literal_eval(sentences)
    for sentence in sentences:
        result = get_aspect_based_sentiment(sentence)
        results.append(result)
    return results

In [18]:
# Deze code berekent het sentiment voor de vertaalde zinnen die het woord "POLICE" bevatten.  

df_en["police_sentences_sentiment"] = df_en.en_police_sentences.progress_apply(lambda x: get_sentence_sentiment(x))

100%|██████████| 5412/5412 [05:21<00:00, 16.84it/s]


In [20]:
df_en.to_csv("df3.csv")